In [1]:
from pathlib import Path
import sys

In [2]:
import pandas as pd
import numpy as np

In [3]:
from imblearn.pipeline import Pipeline
from imblearn import FunctionSampler
from sklearn.preprocessing import RobustScaler, PowerTransformer
from sklearn.svm import OneClassSVM
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    make_scorer ,
    precision_score, recall_score, f1_score
    )

In [4]:
# Enable imports from the parent directory
parent_dir = '..'
sys.path.append(parent_dir)

from utils import draw_pr_curve_from_cv_results

In [5]:
data_dir = Path('../results')
train = pd.read_csv(data_dir / 'train_clean.csv')
test = pd.read_csv(data_dir / 'test_clean.csv')

In [6]:
num_cols = train.dtypes[(train.dtypes == 'float64') | (train.dtypes == 'int64')].index
target_feat = 'hasAwards'

In [7]:
train_X, train_y = train[num_cols], train[target_feat].map({True:-1, False:1})
test_X, test_y = test[num_cols], test[target_feat].map({True:-1, False:1})

In [8]:
train_y.value_counts()

hasAwards
 1    96128
-1     8536
Name: count, dtype: int64

The base idea is to use `FunctionSampler` to remove titles that received awards from the training set and check if they are classified as anomalous on the test set.

In [9]:
def remove_positive_class(X, y):
    return X[y == 1], y[y == 1]

In [10]:
sampler = FunctionSampler(func=remove_positive_class)

In [11]:
estimator = Pipeline([
    ('scaler', RobustScaler()),
    ('sampler', sampler),
    ('svm', OneClassSVM())
])

## Validation

In [12]:
cv = StratifiedKFold(shuffle=True, random_state=42)

In [13]:
scoring = dict(
    precision=make_scorer(precision_score, pos_label=-1),
    recall=make_scorer(recall_score, pos_label=-1),
    f1=make_scorer(f1_score, pos_label=-1)
)

In [14]:
results = cross_validate(
    estimator,
    train_X,
    train_y,
    scoring=scoring,
    n_jobs=-1
)

In [20]:
for key, value in results.items():
    if key.startswith('test'):
        print(f"mean {key.replace('test_', '')}: {np.mean(value):0.2f} ({np.std(value):0.2f})")

mean precision: 0.10 (0.00)
mean recall: 0.62 (0.01)
mean f1: 0.17 (0.00)


## Test

In [15]:
estimator.fit(train_X, train_y)

,steps,"[('scaler', ...), ('sampler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,with_centering,True
,with_scaling,True
,quantile_range,"(25.0, ...)"
,copy,True
,unit_variance,False
,func,<function rem...x7fce42429260>
,accept_sparse,True


In [16]:
y_pred = estimator.predict(test_X)

In [17]:
print(classification_report(y_true=test_y, y_pred=y_pred))

              precision    recall  f1-score   support

          -1       0.10      0.63      0.18      3805
           1       0.94      0.50      0.65     41052

    accuracy                           0.51     44857
   macro avg       0.52      0.56      0.42     44857
weighted avg       0.86      0.51      0.61     44857

